# Step 4.g — SVM climate suitability, ERA5 and the four AMOC scenarios (EC-Earth3 grid, global)

For each crop, the SVM of 3.g (trained on ERA5 1981–2014 over the MIRCA2000 cropland, with the BIO variant
`CHOSEN_VARIANT` and the radiation `CHOSEN_RAD` of the tests) is applied to **every land cell**:
- **ERA5** (1981–2014): the BIO variables of 1.1.d, the reference climate;
- the **four AMOC scenarios** (2025 / 2050 stabilisation, original and hosing): the delta BIO variables of 1.3.a
  (ERA5 + EC-Earth3 change).

**Suitability probability, at equal class priors.** The SVM is trained with balanced class weights (3.g), but the
Platt scaling of libsvm (`predict_proba`) fits its sigmoid on the unweighted training cells, so the probabilities
reproduce the share of presences of each crop in its training table (maize: presences the majority, mean probability
0.78 on the cropland; rice: 0.32). The climate probability is later multiplied by the soil suitability (step 5), so
its scale must mean the same for every crop. With `BALANCED = True` the probability is brought to equal class priors
(prior correction of the odds, Saerens et al. 2002, *Neural Comput.* 14, 21–41; Elkan 2001, *IJCAI*):
logit p' = logit p − log(n_presences / n_absences) (`cu.balanced_proba`). It is a monotonic transform: the ranking of
the cells and the AUC do not change. The Platt probability stays in `suit_proba_platt`.

Binary suitability = probability ≥ `THRESHOLD` (0.5, the neutral point at equal priors), for the climate-only maps
and areas; the binary map of the final product is taken on the climate × soil index in step 5. Areas in Mha of land
(cell area × land fraction of 0.1), and the MIRCA2000 harvested areas (crop, all crops) in the suitable cells.

**Climates outside the training (MESS).** The SVM is trained on the MIRCA2000 cropland only: it has never seen the
climates beyond the edge of today's agriculture (Antarctica, Greenland, central Sahara), and far from the training
cells an RBF SVM gives back its intercept (Antarctica "suitable" for maize). The multivariate environmental
similarity surface (MESS, Elith et al. 2010, *Methods Ecol. Evol.* 1, 330–342) measures, for each cell, how far each
BIO variable is from the values of the training cells of that crop (ERA5 1981–2014 cropland); MESS < 0 = at least one
variable outside the training range ("novel climate"). The MESS looks at one variable at a time: a cell whose
variables are each within their range but whose combination exists in no training cell is not flagged (winter wheat
on the coasts of Greenland in test 4). **NT2** of ExDet (Mesgaran et al. 2014, *Divers. Distrib.* 20, 1147–1159)
flags those: the Mahalanobis distance to the training climates, divided by the largest one among the training cells;
NT2 > 1 = novel combination. Novel = MESS < 0 or NT2 > 1. The same reference (ERA5) for the four scenarios, so a
scenario climate outside today's cultivated climates is flagged too.

Novel cells are **outside the training domain**: the model cannot assess them. They get `suit_proba` 0 (so that a
product never counts them as suitable) and their own class in `suit_class` (0 not suitable, 1 suitable, 2 novel), and
the areas report them apart: a cell that becomes novel in a scenario is not counted as a loss of suitable land
(Elith et al. 2010 recommend flagging extrapolation rather than predicting through it). The hosing effect is also
given on the cells assessable in both scenarios of the pair.

Outputs in `<out_root>/4_svm_predict/<route>/<variant>_<radiation>/`: `suitability_<crop>.nc` (`suit_proba`,
`suit_bin`, `suit_class`, `novel`, `suit_proba_svm`, `suit_proba_platt`, `mess`, `mess_var`, `nt2`, dimension
`scenario`), `suitable_area_<...>.csv`, `amoc_effect_<...>.csv`, `novel_climate_<...>.csv` and maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the choices of the tests of 3.g
CHOSEN_VARIANT = "A_60d"           # test 1
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # test 2; no_rad until the rsds of the hosing runs arrives (as 1.3.a);
                                                     # set for one run with the environment (CHOSEN_RAD=rad_era5)
CELL_AREA_THR = 0.0005            # test 4; None = route no_threshold (tests 1-3); 0.0005 = route cellareafrac, presence and
                                  # background = area > 0.05 % of the cell land area (test 4); same value in 1.1.d, 3.g, 1.3.a, 4.g
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
ERA5_ONLY = os.environ.get("ERA5_ONLY", "0") == "1"   # ERA5 alone (no delta BIO needed: rad_era5 before the hosing rsds)
if ERA5_ONLY:
    RUNS, PAIRS = [], []
BALANCED = True                    # Platt probability brought to equal class priors (cu.balanced_proba); False = Platt
THRESHOLD = 0.5                    # climate-only binary map (the final binary map is taken on climate x soil, step 5)
assert CHOSEN_VARIANT in ("A_60d", "B_season", "C_crop") and CHOSEN_RAD.removesuffix("_relief") in ("rad_era5", "no_rad", "rad_fao56"), \
    "set CHOSEN_VARIANT and CHOSEN_RAD to the choices of the tests of 3.g"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
DELTA = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET.removesuffix("_relief"))   # RELIEF (test 6) is
                                                                                                # static: not in the BIO files
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
OUT = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
CRO = cu.mirca_cropland(CFG, CELL_AREA_THR).values        # training background
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6   # land area of the cells (Mha)
with xr.open_dataset(cu.out_dir(CFG, cu.MASKS) / "mirca2000_any_crop_ecearth3.nc") as ds:
    CROPLAND_HA = np.nan_to_num(ds["area_ha"].values)    # MIRCA2000 harvested area of all crops (ha)
SCEN = {"ERA5": None, **{r: r for r in RUNS}}
print(f"set {SET} | models {MODELS} | balanced probability {BALANCED} | scenarios ERA5 + {RUNS}\noutputs {OUT}")

## 1. Prediction

In [ ]:
def bio_file(scenario, crop):
    if scenario == "ERA5":
        return ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
    return DELTA / f"BIOvariables_delta_{scenario}_{crop}.nc"


def predict(bundle, path, ref):
    # Platt probability, MESS (and its most dissimilar variable), NT2 of every land cell
    with xr.open_dataset(path) as ds:
        X = cu.bio_array(ds, bundle["bio_cols"], CFG).reshape(-1, len(bundle["bio_cols"]))
    ok = np.isfinite(X).all(axis=1) & LAND.ravel()
    p, m, v, n = np.full(X.shape[0], np.nan), np.full(X.shape[0], np.nan), np.full(X.shape[0], -1), np.full(X.shape[0], np.nan)
    p[ok] = bundle["model"].predict_proba(bundle["scaler"].transform(X[ok]))[:, 1]
    m[ok], v[ok] = cu.mess(X[ok], ref)
    n[ok] = cu.nt2(X[ok], ref)
    return p.reshape(LAND.shape), m.reshape(LAND.shape), v.reshape(LAND.shape), n.reshape(LAND.shape)


mha = lambda mask: float(np.nansum(LAND_MHA * mask))
rows, novel_rows, SUIT, NOVEL = [], [], {}, {}
for crop in CROPS:
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    n1, n0 = bundle["n_class1"], bundle["n_class0"]
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(bio_file("ERA5", crop)) as ds:  # training cells: the climates the SVM has seen (from the
        REF = cu.bio_array(ds, bundle["bio_cols"], CFG)[ii, jj]                       # netCDF, not the rounded csv)
    REF = REF[np.isfinite(REF).all(axis=1)]
    R3 = {s: predict(bundle, bio_file(s, crop), REF) for s in SCEN}
    stack = lambda k: xr.DataArray(np.stack([R3[s][k] for s in SCEN]), dims=("scenario", "lat", "lon"),
                                   coords={"scenario": list(SCEN), "lat": GRID["lat"], "lon": GRID["lon"]})
    p_platt, mss, mvar, nt = stack(0), stack(1), stack(2), stack(3)
    p_svm = cu.balanced_proba(p_platt, n1, n0) if BALANCED else p_platt
    novel = ((mss < 0) | (nt > 1)) & p_svm.notnull()
    proba = xr.where(novel, 0.0, p_svm)                   # novel climate -> 0 (NaN off land stays NaN)
    suit = xr.where(proba.notnull(), (proba >= THRESHOLD).astype("float32"), np.nan)
    cls = xr.where(proba.notnull(), xr.where(novel, 2.0, suit), np.nan)
    xr.Dataset({"suit_proba": proba.astype("float32").assign_attrs(
                    long_name="climate suitability probability" + (", equal class priors" if BALANCED else ", Platt"),
                    note="0 where the climate is novel (MESS < 0 or NT2 > 1), see suit_class"),
                "suit_bin": suit.assign_attrs(long_name="binary climate suitability", threshold=THRESHOLD),
                "suit_class": cls.astype("float32").assign_attrs(long_name="0 not suitable, 1 suitable, 2 novel climate (outside the training domain)"),
                "novel": novel.astype("int8").assign_attrs(long_name="novel climate: MESS < 0 or NT2 > 1"),
                "suit_proba_svm": p_svm.astype("float32").assign_attrs(long_name="SVM probability, no MESS",
                                                                       balanced=str(BALANCED)),
                "suit_proba_platt": p_platt.astype("float32").assign_attrs(long_name="Platt probability of libsvm (prior = share of presences of the training table)"),
                "mess": mss.astype("float32").assign_attrs(long_name="MESS (Elith et al. 2010), reference ERA5 training cells",
                                                           note="< 0: at least one BIO variable outside the training range"),
                "mess_var": mvar.astype("int16").assign_attrs(long_name="most dissimilar variable (index in bio_cols)",
                                                              bio_cols=", ".join(bundle["bio_cols"]), missing=-1),
                "nt2": nt.astype("float32").assign_attrs(long_name="NT2 of ExDet (Mesgaran et al. 2014), reference ERA5 training cells",
                                                         note="> 1: combination of variables outside the training cloud")},
               attrs={"crop": crop, "set": SET, "model": f"one-pass SVM, cost {bundle['best_cost']}, gamma {bundle['best_gamma']}",
                      "bio_cols": ", ".join(bundle["bio_cols"]), "presences": n1, "absences": n0,
                      "balanced": str(BALANCED)}).to_netcdf(OUT / f"suitability_{crop}.nc")
    # in-sample check on the training cells (not a validation: the cross-validation of 3.g is)
    auc_in = roc_auc_score(tr.label.values, proba.sel(scenario="ERA5").values[ii, jj])
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    SUIT[crop] = {s: suit.sel(scenario=s).values == 1 for s in SCEN}
    NOVEL[crop] = {s: novel.sel(scenario=s).values for s in SCEN}
    era5_area = mha(SUIT[crop]["ERA5"])
    for s in SCEN:
        b, nv = SUIT[crop][s], NOVEL[crop][s]
        a = mha(b)
        rows.append({"crop": crop, "scenario": s, "presence share": n1 / (n1 + n0),
                     "suitable land (Mha)": a,
                     "change vs ERA5 (%)": 100 * (a - era5_area) / era5_area if s != "ERA5" else 0.0,
                     "suitable land in cropland cells (Mha)": mha(b & CRO),
                     "MIRCA cropland harvested area in suitable cells (Mha)": CROPLAND_HA[b].sum() / 1e6,
                     "MIRCA area of the crop in suitable cells (%)": 100 * crop_ha[b].sum() / crop_ha.sum(),
                     "novel climate (Mha)": mha(nv),
                     "novel in cropland cells (Mha)": mha(nv & CRO),
                     "MIRCA area of the crop in novel cells (%)": 100 * crop_ha[nv].sum() / crop_ha.sum(),
                     "SVM alone, no MESS (Mha)": mha(p_svm.sel(scenario=s).values >= THRESHOLD),
                     "mean probability on cropland": float(np.nanmean(proba.sel(scenario=s).values[CRO])),
                     "mean Platt probability on cropland": float(np.nanmean(p_platt.sel(scenario=s).values[CRO])),
                     "in-sample AUC (ERA5)": auc_in if s == "ERA5" else np.nan})
        # which variable puts the cells outside the training range (MESS < 0), and novel combinations only (NT2)
        v, mneg = mvar.sel(scenario=s).values, mss.sel(scenario=s).values < 0
        nt2_only = nv & ~mneg
        if nt2_only.any():
            novel_rows.append({"crop": crop, "scenario": s, "variable": "NT2 > 1 (combination)",
                               "novel area (Mha)": mha(nt2_only), "of which in cropland cells (Mha)": mha(nt2_only & CRO)})
        for k, c in enumerate(bundle["bio_cols"]):
            sel = mneg & (v == k)
            if sel.any():
                novel_rows.append({"crop": crop, "scenario": s, "variable": c, "novel area (Mha)": mha(sel),
                                   "of which in cropland cells (Mha)": mha(sel & CRO)})
    print(f"{crop}: presences {n1 / (n1 + n0):.0%}, ERA5 {era5_area:.0f} Mha suitable, in-sample AUC {auc_in:.3f}, "
          f"mean probability on cropland {rows[-len(SCEN)]['mean probability on cropland']:.2f} "
          f"(Platt {rows[-len(SCEN)]['mean Platt probability on cropland']:.2f})", flush=True)
area = pd.DataFrame(rows).set_index(["crop", "scenario"])
area.to_csv(OUT / f"suitable_area_{SET}.csv")
novel_tab = pd.DataFrame(novel_rows, columns=["crop", "scenario", "variable", "novel area (Mha)",
                                              "of which in cropland cells (Mha)"]).set_index(["crop", "scenario", "variable"])
novel_tab.to_csv(OUT / f"novel_climate_{SET}.csv")
with pd.option_context("display.max_rows", 200, "display.max_columns", 30, "display.width", 250):
    display(area.round(2))
    print("novel climate: MESS < 0 by most dissimilar variable, and NT2 > 1 only"); display(novel_tab.round(2))

## 2. Effect of the AMOC collapse: hosing − original, per stabilisation level

On all land (novel cells count as not suitable) and on the cells assessable in both scenarios of the pair (novel in
neither): the second is the effect the model can actually tell.

In [ ]:
if not PAIRS:
    print("ERA5 only: no AMOC pair")
else:
    rows = []
    for crop in CROPS:
        for hos, orig in PAIRS:
            a_h, a_o = mha(SUIT[crop][hos]), mha(SUIT[crop][orig])
            both = ~(NOVEL[crop][hos] | NOVEL[crop][orig])
            b_h, b_o = mha(SUIT[crop][hos] & both), mha(SUIT[crop][orig] & both)
            rows.append({"crop": crop, "pair": f"{hos} - {orig}", "area change (Mha)": a_h - a_o,
                         "area change (%)": 100 * (a_h - a_o) / a_o if a_o else np.nan,
                         "area change, assessable in both (Mha)": b_h - b_o,
                         "area change, assessable in both (%)": 100 * (b_h - b_o) / b_o if b_o else np.nan,
                         "novel in either (Mha)": mha(~both & LAND)})
    amoc = pd.DataFrame(rows).set_index(["crop", "pair"])
    amoc.to_csv(OUT / f"amoc_effect_{SET}.csv")
    with pd.option_context("display.width", 250):
        display(amoc.round(2))

    grey = ListedColormap(["#9e9e9e"])
    for crop in CROPS:
        ds = xr.open_dataset(OUT / f"suitability_{crop}.nc")
        nov = ds.novel == 1
        fig, axes = plt.subplots(2, 3, figsize=(20, 7))
        for ax, s in zip(axes[0], ["ERA5", PAIRS[0][1], PAIRS[1][1]]):
            ds.suit_proba.sel(scenario=s).where(~nov.sel(scenario=s)).plot(ax=ax, vmin=0, vmax=1, cmap="YlGn")
            nov.sel(scenario=s).where(nov.sel(scenario=s)).plot(ax=ax, cmap=grey, add_colorbar=False)
            ax.set_title(f"{crop}: {s} (grey: novel climate)", fontsize=9)
        ds.mess.sel(scenario="ERA5").clip(-100, 100).plot(ax=axes[1, 0], vmin=-100, vmax=100, cmap="PuOr")
        axes[1, 0].set_title(f"{crop}: MESS ERA5 (< 0 = novel climate)", fontsize=9)
        for ax, (hos, orig) in zip(axes[1, 1:], PAIRS):
            either = nov.sel(scenario=hos) | nov.sel(scenario=orig)
            (ds.suit_proba.sel(scenario=hos) - ds.suit_proba.sel(scenario=orig)).where(~either).plot(ax=ax, vmin=-0.5, vmax=0.5, cmap="RdBu")
            either.where(either).plot(ax=ax, cmap=grey, add_colorbar=False)
            ax.set_title(f"{hos} - {orig} (grey: novel in either)", fontsize=9)
        for ax in axes.ravel():
            ax.set_xlabel(""); ax.set_ylabel("")
        plt.tight_layout(); plt.savefig(OUT / f"suitability_maps_{SET}_{crop}.png", dpi=80); plt.show()